# Phase 3 — Full ImageNette Policy Comparison

**Run on Google Colab with GPU runtime.**

Same as Phase 2 but on the full ImageNette val set (~3900 images) instead of 100.

Design decisions to prevent data loss:
- One cell per policy. Each saves its own parquet to Drive immediately.
- If a cell crashes or the session dies, restart and skip already-saved policies.
- Analysis cells load from Drive files — they do NOT depend on in-memory state from policy cells.
- Each policy cell prints a skip message if its parquet already exists.

Policies:
- `random`, `center`, `coverage`
- `saliency_lowres`, `saliency_ior`, `saliency_ior_tuned` (strength=2.0, radius=0.4)
- `inverse_saliency`
- `ORACLE_saliency_fullres` (diagnostic only)

Expected runtime per policy on T4: ~25–40 min. Total: ~4–5 hrs.

## 3.0 — Setup (run once at session start)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['HF_HOME'] = '/content/drive/MyDrive/canvit_cache'
RESULTS_DIR = '/content/drive/MyDrive/canvit_results/phase3'
os.makedirs(RESULTS_DIR, exist_ok=True)

REPO_URL = 'https://github.com/johnsaurabh/active-canvit-gaze.git'
REPO_DIR = '/content/active-canvit-gaze'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# Reload saliency modules after pull to pick up the scipy fix
import importlib
import importlib.util

!pip install -q 'canvit-pytorch @ git+https://github.com/m2b3/CanViT-PyTorch.git'
!pip install -q huggingface_hub
print('Setup complete.')

## 3.1 — Load model

In [ ]:
import torch
import torch.nn.functional as F
from canvit_pytorch import CanViTForImageClassification
from canvit_pytorch import Viewpoint as CanViTViewpoint
from canvit_pytorch import sample_at_viewpoint
from canvit_pytorch.preprocess import preprocess

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CHECKPOINT = 'canvit/canvitb16-add-vpe-finetune-g128px-s512px-in1k-2026-04-06'
model = CanViTForImageClassification.from_pretrained(CHECKPOINT).eval().to(DEVICE)
print('Model loaded.')

CANVAS_GRID_SIZE = 32
GLIMPSE_SIZE_PX  = 128
SCENE_SIZE       = 512
LOCAL_SCALE      = 0.25
GLIMPSE_BUDGETS  = [0, 1, 2, 3, 4, 5, 6, 8]
N_LOCAL          = max(GLIMPSE_BUDGETS)

## 3.2 — Load full ImageNette val set

In [ ]:
import json, requests
from pathlib import Path

DATA_DIR       = '/content/drive/MyDrive/data'
IMAGENETTE_DIR = os.path.join(DATA_DIR, 'imagenette2-320')
VAL_DIR        = os.path.join(IMAGENETTE_DIR, 'val')
assert os.path.exists(VAL_DIR), f'ImageNette val not found at {VAL_DIR}.'

HEADERS = {'User-Agent': 'active-canvit-gaze/1.0'}
class_idx = json.loads(requests.get(
    'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json',
    headers=HEADERS, timeout=10).text)
synset_to_idx = {v[0]: int(k) for k, v in class_idx.items()}

all_samples = []
for synset in sorted(os.listdir(VAL_DIR)):
    synset_dir = Path(VAL_DIR) / synset
    if not synset_dir.is_dir():
        continue
    label = synset_to_idx.get(synset, -1)
    for img_path in sorted(synset_dir.iterdir()):
        if img_path.suffix.lower() in {'.jpeg', '.jpg', '.png'}:
            all_samples.append({'path': str(img_path), 'synset': synset, 'label': label})

print(f'Total val images: {len(all_samples)}')
transform = preprocess(SCENE_SIZE)

## 3.3 — Define runner and helper (run once)

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from policies.base import Viewpoint as PolicyViewpoint

LOWRES_SIZE = 64

def run_policy_sequence(policy, image_tensor, n_local=8):
    lowres = F.interpolate(
        image_tensor, size=(LOWRES_SIZE, LOWRES_SIZE),
        mode='bilinear', align_corners=False
    )
    policy.reset()
    state = model.init_state(batch_size=1, canvas_grid_size=CANVAS_GRID_SIZE)
    all_logits, history = [], []

    with torch.inference_mode():
        vp0 = PolicyViewpoint(x=0.0, y=0.0, s=1.0)
        centers, scales = vp0.to_canvit(DEVICE)
        cvp = CanViTViewpoint(centers=centers, scales=scales)
        glimpse = sample_at_viewpoint(spatial=image_tensor, viewpoint=cvp, glimpse_size_px=GLIMPSE_SIZE_PX)
        logits, state = model(glimpse=glimpse, state=state, viewpoint=cvp)
        all_logits.append(logits.cpu())
        history.append(vp0)

        for _ in range(n_local):
            vp = policy.select_next(
                viewpoint_history=history,
                model_state=state,
                lowres_preview=lowres,
                full_image=image_tensor,
            )
            centers, scales = vp.to_canvit(DEVICE)
            cvp = CanViTViewpoint(centers=centers, scales=scales)
            glimpse = sample_at_viewpoint(spatial=image_tensor, viewpoint=cvp, glimpse_size_px=GLIMPSE_SIZE_PX)
            logits, state = model(glimpse=glimpse, state=state, viewpoint=cvp)
            all_logits.append(logits.cpu())
            history.append(vp)

    return all_logits, history


def run_and_save(policy_name, policy):
    """Run one policy on all_samples and save results to Drive immediately."""
    out_path = os.path.join(RESULTS_DIR, f'phase3_{policy_name}.parquet')
    if os.path.exists(out_path):
        print(f'SKIP {policy_name} — already saved at {out_path}')
        return

    print(f'\n--- {policy_name} ({len(all_samples)} images) ---')
    records = []
    n_failed = 0

    for sample in tqdm(all_samples, desc=policy_name, leave=True):
        try:
            img = Image.open(sample['path']).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(DEVICE)
            all_logits, history = run_policy_sequence(policy, img_tensor, n_local=N_LOCAL)

            for t, logits in enumerate(all_logits):
                probs = torch.softmax(logits, dim=-1)
                pred  = int(probs.argmax())
                top5  = torch.topk(probs, k=5, dim=-1).indices[0].tolist()
                records.append({
                    'policy':       policy_name,
                    'image_id':     sample['path'],
                    'synset':       sample['synset'],
                    'true_label':   sample['label'],
                    'timestep':     t,
                    'pred_label':   pred,
                    'confidence':   float(probs.max()),
                    'correct_top1': int(pred == sample['label']),
                    'correct_top5': int(sample['label'] in top5),
                    'vp_x':         history[t].x,
                    'vp_y':         history[t].y,
                    'vp_s':         history[t].s,
                })
        except Exception as e:
            n_failed += 1
            if n_failed <= 3:
                print(f'  FAIL: {sample["path"]}: {e}')

    if not records:
        print(f'  ERROR: all images failed for {policy_name}. NOT saved.')
        return

    df = pd.DataFrame(records)
    df.to_parquet(out_path, index=False)

    budget_df = df[df['timestep'].isin(GLIMPSE_BUDGETS)]
    acc_by_t  = budget_df.groupby('timestep')['correct_top1'].mean()
    print(f'  T=0: {acc_by_t.get(0, float("nan")):.3f}   '
          f'T=8: {acc_by_t.get(8, float("nan")):.3f}   '
          f'failed: {n_failed}/{len(all_samples)}')
    print(f'  Saved {len(df)} records → {out_path}')

print('Runner and helper defined.')

## 3.4 — Instantiate policies

In [ ]:
import importlib
import policies.saliency_policy as _sal_mod
import policies.saliency_ior_policy as _ior_mod
importlib.reload(_sal_mod)
importlib.reload(_ior_mod)

from policies.random_policy       import RandomPolicy
from policies.center_policy       import CenterPolicy
from policies.coverage_policy     import CoveragePolicy
from policies.saliency_policy     import LowResSaliencyPolicy
from policies.saliency_ior_policy import SaliencyIORPolicy
from policies.negative_control    import InverseSaliencyPolicy
from policies.oracle_policy       import FullResSaliencyOracle

policy_random           = RandomPolicy(scale=LOCAL_SCALE, seed=0)
policy_center           = CenterPolicy(scale=LOCAL_SCALE)
policy_coverage         = CoveragePolicy(scale=LOCAL_SCALE)
policy_saliency         = LowResSaliencyPolicy(scale=LOCAL_SCALE)
policy_ior              = SaliencyIORPolicy(scale=LOCAL_SCALE)
policy_ior_tuned        = SaliencyIORPolicy(scale=LOCAL_SCALE, ior_strength=2.0, ior_radius=0.4, ior_decay=0.85)
policy_inverse          = InverseSaliencyPolicy(scale=LOCAL_SCALE)
policy_oracle           = FullResSaliencyOracle(scale=LOCAL_SCALE)

print('All policies instantiated.')

## 3.5 — Run policies
Each cell is independent. If a cell already saved its parquet it will skip automatically.
Run them in order, or resume from where you left off after a session restart.

In [ ]:
# --- RANDOM ---
run_and_save('random', policy_random)

In [ ]:
# --- CENTER ---
run_and_save('center', policy_center)

In [ ]:
# --- COVERAGE ---
run_and_save('coverage', policy_coverage)

In [ ]:
# --- SALIENCY LOWRES ---
run_and_save('saliency_lowres', policy_saliency)

In [ ]:
# --- SALIENCY IOR (original parameters) ---
run_and_save('saliency_ior', policy_ior)

In [ ]:
# --- SALIENCY IOR TUNED (strength=2.0, radius=0.4) ---
run_and_save('saliency_ior_tuned', policy_ior_tuned)

In [ ]:
# --- INVERSE SALIENCY (negative control) ---
run_and_save('inverse_saliency', policy_inverse)

In [ ]:
# --- ORACLE (diagnostic only) ---
run_and_save('ORACLE_saliency_fullres', policy_oracle)

## 3.6 — Load all results from Drive
Run this after all policy cells above are done (or after resuming a session).
This cell loads from Drive files — it does not depend on any in-memory state.

In [ ]:
import os, pandas as pd, numpy as np

POLICY_ORDER = [
    'random', 'center', 'coverage',
    'saliency_lowres', 'saliency_ior', 'saliency_ior_tuned',
    'inverse_saliency', 'ORACLE_saliency_fullres',
]
ORACLE_POLICIES = {'ORACLE_saliency_fullres'}
GLIMPSE_BUDGETS = [0, 1, 2, 3, 4, 5, 6, 8]
RESULTS_DIR = '/content/drive/MyDrive/canvit_results/phase3'

dfs = {}
for name in POLICY_ORDER:
    path = os.path.join(RESULTS_DIR, f'phase3_{name}.parquet')
    if os.path.exists(path):
        dfs[name] = pd.read_parquet(path)
        n_images = dfs[name]['image_id'].nunique()
        print(f'  Loaded {name}: {n_images} images')
    else:
        print(f'  MISSING: {name} — run its cell above first')

# Build per-policy correct matrices (N_images x N_budgets)
per_policy_correct = {}
per_policy_vp_seqs = {}

for name, df in dfs.items():
    budget_df = df[df['timestep'].isin(GLIMPSE_BUDGETS)]
    pivot = budget_df.pivot_table(
        index='image_id', columns='timestep', values='correct_top1'
    ).reindex(columns=GLIMPSE_BUDGETS)
    per_policy_correct[name] = pivot.values.astype(np.float32)

    # Reconstruct viewpoint sequences (all timesteps, for spatial metrics)
    seqs = []
    for img_id, grp in df.groupby('image_id'):
        grp = grp.sort_values('timestep')
        seqs.append(list(zip(grp['vp_x'].tolist(), grp['vp_y'].tolist())))
    per_policy_vp_seqs[name] = seqs

print(f'\nLoaded {len(dfs)}/{len(POLICY_ORDER)} policies.')

## 3.7 — Compute AUGC

In [ ]:
from evaluation.metrics import compute_accuracy_curve, paired_bootstrap_ci, compute_spatial_metrics

augc_results = {}
for name, correct in per_policy_correct.items():
    mean_acc, per_img_augc = compute_accuracy_curve(correct, GLIMPSE_BUDGETS)
    augc_results[name] = (mean_acc, per_img_augc)

print(f'{"Policy":<28} {"AUGC":>8}')
print('-' * 38)
for name in POLICY_ORDER:
    if name not in augc_results:
        continue
    _, per_img = augc_results[name]
    tag = ' [ORACLE]' if name in ORACLE_POLICIES else ''
    print(f'{name:<28} {per_img.mean():>8.4f}{tag}')

## 3.8 — Accuracy-vs-Glimpses plot

In [ ]:
import matplotlib.pyplot as plt

COLORS = {
    'random':                  '#888888',
    'center':                  '#4e79a7',
    'coverage':                '#f28e2b',
    'saliency_lowres':         '#59a14f',
    'saliency_ior':            '#e15759',
    'saliency_ior_tuned':      '#d62728',
    'inverse_saliency':        '#b07aa1',
    'ORACLE_saliency_fullres':  '#76b7b2',
}

valid_policies  = [n for n in POLICY_ORDER if n not in ORACLE_POLICIES and n in augc_results]
oracle_policies = [n for n in POLICY_ORDER if n in ORACLE_POLICIES and n in augc_results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, include_oracle in zip(axes, [False, True]):
    for name in valid_policies:
        mean_acc, _ = augc_results[name]
        ls = '--' if name == 'inverse_saliency' else '-'
        ax.plot(GLIMPSE_BUDGETS, mean_acc, ls, color=COLORS[name], marker='o', ms=4, label=name)
    if include_oracle:
        for name in oracle_policies:
            mean_acc, _ = augc_results[name]
            ax.plot(GLIMPSE_BUDGETS, mean_acc, ':', color=COLORS[name], marker='s', ms=4,
                    linewidth=1.5, label=f'{name} [ORACLE]')
    title = 'All policies (oracle dotted)' if include_oracle else 'Valid policies only'
    ax.set_title(f'{title}\n(ImageNette full val, N≈3900)')
    ax.set_xlabel('Glimpse budget')
    ax.set_ylabel('Top-1 accuracy')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'phase3_policy_curves.png')
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

## 3.9 — Paired bootstrap CI vs random

In [ ]:
_, random_augc = augc_results['random']

print('Paired bootstrap CI — AUGC diff vs random (valid policies only)')
print()
print(f'{"Policy":<28} {"AUGC":>8}  {"Diff":>8}  {"95% CI":>22}  Result')
print('-' * 80)

summary_rows = []
for name in valid_policies:
    _, per_img = augc_results[name]
    diff, lo, hi = paired_bootstrap_ci(per_img, random_augc, n_bootstrap=10_000)
    beats = 'YES' if lo > 0 else ('NO' if hi < 0 else 'uncertain')
    print(f'{name:<28} {per_img.mean():>8.4f}  {diff:>+8.4f}  [{lo:>+8.4f}, {hi:>+8.4f}]  {beats}')
    summary_rows.append({'policy': name, 'mean_augc': float(per_img.mean()),
                         'diff_vs_random': diff, 'ci_lower': lo, 'ci_upper': hi,
                         'beats_random': beats, 'type': 'valid'})

print()
print('ORACLE (diagnostic only):')
for name in oracle_policies:
    _, per_img = augc_results[name]
    print(f'  {name}: AUGC = {per_img.mean():.4f}')
    summary_rows.append({'policy': name, 'mean_augc': float(per_img.mean()),
                         'diff_vs_random': float('nan'), 'ci_lower': float('nan'),
                         'ci_upper': float('nan'), 'beats_random': 'N/A (ORACLE)', 'type': 'oracle'})

summary_df = pd.DataFrame(summary_rows)

## 3.10 — Spatial behaviour metrics

In [ ]:
print(f'{"Policy":<28} {"Mean displ":>12}  {"Revisit rate":>14}  {"Mean |center|":>14}')
print('-' * 72)

for name in POLICY_ORDER:
    if name not in per_policy_vp_seqs:
        continue
    m = compute_spatial_metrics(per_policy_vp_seqs[name])
    tag = ' [ORACLE]' if name in ORACLE_POLICIES else ''
    print(
        f'{name:<28} '
        f'{m.get("mean_displacement", float("nan")):>12.4f}  '
        f'{m.get("revisit_rate", float("nan")):>14.4f}  '
        f'{m.get("mean_center_distance", float("nan")):>14.4f}{tag}'
    )

## 3.11 — Save summary and print final verdict

In [ ]:
summary_path = os.path.join(RESULTS_DIR, 'phase3_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f'Summary saved to {summary_path}')

print()
print('=' * 60)
print('PHASE 3 — FINAL SUMMARY')
print('=' * 60)
print(f'Dataset:    ImageNette full val (N≈3900). Dev only — NOT ImageNet-1k.')
print(f'Checkpoint: canvit/canvitb16-add-vpe-finetune-g128px-s512px-in1k-2026-04-06')
print()

ranked = sorted(
    [(n, float(augc_results[n][1].mean())) for n in valid_policies if n in augc_results],
    key=lambda x: -x[1]
)
print('Valid policy ranking (by AUGC):')
for rank, (name, augc) in enumerate(ranked, 1):
    row = summary_df[summary_df['policy'] == name].iloc[0]
    print(f'  {rank}. {name:<28} AUGC={augc:.4f}  diff={row["diff_vs_random"]:+.4f}  '
          f'95%CI=[{row["ci_lower"]:+.4f}, {row["ci_upper"]:+.4f}]  {row["beats_random"]}')

print()
for name in oracle_policies:
    if name in augc_results:
        print(f'ORACLE {name}: AUGC={augc_results[name][1].mean():.4f} [diagnostic only]')